# OCR recognition fine-tuning smoke test (Colab)


## 1. GPU и установка (только для обучения §6–§10)

In [ ]:
!nvidia-smi

In [1]:
# Colab по умолчанию: numpy 2.x + opencv-python-headless 4.13.
# PaddleOCR train.py требует numpy 1.26 + opencv 4.6.
%pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python numpy
%pip install -q "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"
%pip install -q "pillow==11.1.0" "pyyaml>=6.0" "pandas==2.2.2"
%pip uninstall -y paddlepaddle paddlepaddle-gpu
%pip install -q "paddlepaddle-gpu==3.2.2" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
print("Готово. Сейчас: Runtime -> Restart session, затем ячейка 2 и дальше.")

Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 MB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [3]:
!pip install -q "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"
!pip install -q "paddlepaddle-gpu==3.3.0" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 445.7 kB/s eta 0:00:00


## 2. Импорты и поиск ручной разметки

In [1]:
import json
import random
import re
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

import pandas as pd
from google.colab import drive

try:
    import paddle
    _paddle_version = paddle.__version__
except ImportError:
    paddle = None
    _paddle_version = 'not installed'

def fix_train_deps() -> None:
    """Re-pin numpy/opencv for PaddleOCR tools/infer."""
    cmd = (
        'pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python numpy && '
        'pip install -q "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"'
    )
    subprocess.run(cmd, shell=True, check=True)


def ensure_paddle_gpu() -> None:
    """Paddle для inference (§11), если §1 не запускали."""
    global paddle, _paddle_version
    try:
        import paddle as _paddle
        if tuple(int(x) for x in _paddle.__version__.split('.')[:2]) >= (3, 0):
            paddle = _paddle
            _paddle_version = _paddle.__version__
            return
    except ImportError:
        pass
    print('Ставлю paddlepaddle-gpu 3.2.2...')
    subprocess.run(
        'pip uninstall -y paddlepaddle paddlepaddle-gpu && '
        'pip install -q "paddlepaddle-gpu==3.2.2" '
        '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/',
        shell=True,
        check=True,
    )
    import paddle as _paddle
    paddle = _paddle
    _paddle_version = _paddle.__version__


def verify_train_deps() -> None:
    code = 'import numpy as np; import cv2; print("OK", np.__version__, cv2.__version__)'
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError('numpy/opencv не подходят — §11 вызовет fix_train_deps(); для обучения нужна §1 + restart.')

drive.mount('/content/drive')

REC_MODEL_NAME = 'cyrillic_PP-OCRv5_mobile_rec'
PROJECT_ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/Avar OCR ANN')
PADDLEOCR_DIR = PROJECT_ROOT / 'PaddleOCR'
# print(PADDLEOCR_DIR)
DATASET_DIR = PROJECT_ROOT / 'dataset' / 'avar_rec_manual'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'rec_avar_v5_smoke'
INFER_DIR = PROJECT_ROOT / 'output' / 'cyrillic_PP-OCRv5_mobile_rec_avar_infer'

for path in [DATASET_DIR, OUTPUT_DIR, INFER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('model:', REC_MODEL_NAME)
print('paddle:', _paddle_version)
if paddle is not None:
    print('cuda compiled:', paddle.is_compiled_with_cuda())
    print('gpu count:', paddle.device.cuda.device_count())

Error: Can not import paddle core while this file exists: /usr/local/lib/python3.12/dist-packages/paddle/base/libpaddle.so


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
model: cyrillic_PP-OCRv5_mobile_rec
paddle: not installed


In [2]:
manual_files = sorted((DRIVE_ROOT / 'classical_layout').rglob('manual_corrections.md'))
if not manual_files:
    raise FileNotFoundError('Не нашла manual_corrections.md в Drive/classical_layout')

for i, path in enumerate(manual_files):
    print(i, path)

MANUAL_INDEX = 0
manual_md_path = manual_files[MANUAL_INDEX]
page_dir = manual_md_path.parent
ocr_json_path = page_dir / 'ocr_results.json'

print('manual:', manual_md_path)
print('page_dir:', page_dir)
print('ocr json:', ocr_json_path, ocr_json_path.exists())

0 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components/manual_corrections.md
manual: /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components/manual_corrections.md
page_dir: /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components
ocr json: /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components/ocr_results.json True


## 3. Парсинг `manual_corrections.md` и сбор dataset

Берём только блок `Corrected`. Каждый crop становится одной OCR recognition training sample.

In [3]:
def refresh_manual_path(current_path: Path) -> Path:
    """Re-resolve manual_corrections.md in case Google Drive state changed."""
    if current_path.exists():
        return current_path

    print('manual_md_path не найден, ищу заново в Drive...')
    refreshed = sorted((DRIVE_ROOT / 'classical_layout').rglob('manual_corrections.md'))
    if refreshed:
        for i, path in enumerate(refreshed):
            print(i, path)
        return refreshed[0]

    raise FileNotFoundError(
        f'Не найден manual_corrections.md. Проверьте, что файл есть на Drive и Drive смонтирован: {current_path}'
    )


def parse_manual_corrections(md_path: Path) -> list[dict]:
    md_path = refresh_manual_path(md_path)
    print('Reading:', md_path)
    print('Exists:', md_path.exists())

    text = md_path.read_text(encoding='utf-8')
    pattern = re.compile(
        r'## block_(\d+).*?Crop: `([^`]+)`.*?Corrected:\s*```text\n(.*?)\n```',
        re.S,
    )
    records = []
    for match in pattern.finditer(text):
        order = int(match.group(1))
        crop_rel = match.group(2).strip()
        corrected = match.group(3).strip()
        if not corrected:
            continue

        crop_path = md_path.parent / crop_rel
        if not crop_path.exists():
            # Fallback: if metadata was moved, resolve by crop filename in local crops folder.
            crop_path = md_path.parent / 'crops' / Path(crop_rel).name

        if not crop_path.exists():
            print('skip missing crop:', crop_path)
            continue

        records.append({
            'order': order,
            'crop_path': crop_path,
            'corrected_text': corrected,
        })
    return sorted(records, key=lambda r: r['order'])

manual_md_path = refresh_manual_path(manual_md_path)
page_dir = manual_md_path.parent
ocr_json_path = page_dir / 'ocr_results.json'

records = parse_manual_corrections(manual_md_path)
print('records:', len(records))
pd.DataFrame([{'order': r['order'], 'crop': r['crop_path'].name, 'text': r['corrected_text'][:80]} for r in records]).head(10)

Reading: /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components/manual_corrections.md
Exists: True
records: 20


,order,crop,text
0,0,block_000.png,"№8, май 2016 сон"
1,1,block_001.png,*\nСахлъи букIуна цIунизе лъарасул
2,2,block_002.png,"PЕГА-РАХЪИНАЛЪУЛ,"
3,3,block_003.png,ВАГЪА-ВАЧАРИЯЛЪУЛ
4,4,block_004.png,Цадахъ кваназе рекъо-\nларел тIагIамазул баян
5,5,block_005.png,ЦӀибил гIакдал нахгун цадахъ\nкванаялъ черхалъ...
6,6,block_006.png,гьуинаб лимон кванаялъ берал унти-\nзарула. Са...
7,7,block_007.png,"ла, гьорола.\nГIемер пер кванаялъ (40 къо-\nял..."
8,8,block_008.png,Вагъа-вачари
9,9,block_009.png,Щибаб квараб квандаса кигIан\nдагьаб букIаниги...


In [4]:
# Build PaddleOCR recognition dataset: images/*.png + train_list.txt / val_list.txt
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
(DATASET_DIR / 'images').mkdir(parents=True, exist_ok=True)

def normalize_label(text: str) -> str:
    """PaddleOCR label file = one line per sample: path<TAB>text."""
    text = text.replace('\t', ' ').replace('\r', ' ')
    text = ' '.join(text.split())
    return text.strip()

label_entries = []
for rec in records:
    dst_name = f"{manual_md_path.parent.parent.name}_block_{rec['order']:03d}.png"
    dst = DATASET_DIR / 'images' / dst_name
    shutil.copy2(rec['crop_path'], dst)
    rel = dst.relative_to(DATASET_DIR).as_posix()
    label_entries.append((rel, normalize_label(rec['corrected_text']), rec['order']))

random.seed(42)
random.shuffle(label_entries)
VAL_RATIO = 0.2
val_n = max(1, int(len(label_entries) * VAL_RATIO)) if len(label_entries) > 1 else 1
val_entries = label_entries[:val_n]
train_entries = label_entries[val_n:] if len(label_entries) > 1 else label_entries
if not train_entries:
    train_entries = val_entries

def write_list(path: Path, entries: list[tuple[str, str, int]]) -> None:
    lines = [f'{rel}\t{text}' for rel, text, _ in entries if text]
    path.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')

write_list(DATASET_DIR / 'train_list.txt', train_entries)
write_list(DATASET_DIR / 'val_list.txt', val_entries)
write_list(DATASET_DIR / 'all_list.txt', label_entries)

for name in ['train_list.txt', 'val_list.txt']:
    p = DATASET_DIR / name
    bad = [ln for ln in p.read_text(encoding='utf-8').splitlines() if ln and '\t' not in ln]
    if bad:
        raise ValueError(f'{name}: {len(bad)} broken lines without TAB')

print('DATASET_DIR:', DATASET_DIR)
print('train:', len(train_entries), 'val:', len(val_entries), 'all:', len(label_entries))
print('sample:', (DATASET_DIR / 'train_list.txt').read_text(encoding='utf-8').splitlines()[:2])

DATASET_DIR: /content/dataset/avar_rec_manual
train: 16 val: 4 all: 20
sample: ['images/0_tlyarata_8_-na_pechat_page_003_block_009.png\tЩибаб квараб квандаса кигIан дагьаб букIаниги къадар зарали- яб жоялъул хутIула кванирукъалда жаниб. Цо хасал гIужаз вагъа-вача- ри гьабичIого рес гьечIо, гьел зара- лиял жал кванирукъалда данделъ- ун чIечIого къватIир рачIинелъун. Гьенир гьел хутIиялъулъ кӀудияб зарал бyго. Гьединлъидал, гьор- кьохъеб къагIидаялъ вагъа-вача- ри (спорт) сахлъи цӀуниялъе бищун лъикIал сабабаздасан буго. Щайгу- релъул, гьелъ лугби хинлъизарула, заралиял жал къватIире инарула ва черхалъе тIадагьлъи щола. Гьелъул гӀужги кванирукъалдаса тӀагIам биун гIодобе биччараб заман буго. Гьелъ- ие къадарги (квен бихIиналъе) 5-6 сагIат букIуна. Гьоркьохъеблъун кко- ла - тIом багIарлъун гIетӀ кIанцIизе', 'images/0_tlyarata_8_-na_pechat_page_003_block_014.png\tму цIикIкӀинабизе бокьани гьеб хIалтIизабизе ккола, рекIехъе жал лъазарун, зикру-пикру гIемер гьабун. Гьединго, каранзуй гIемер 

## 4. Качество исходной OCR-модели до fine-tuning

Сравниваем исходный `ocr_results.json` с ручным `Corrected` и считаем:

- `CER` — character error rate;
- `WER` — word error rate.

Это baseline до обучения. Если `ocr_results.json` получен моделью `cyrillic_PP-OCRv5_mobile_rec`, метрика относится именно к ней.

In [ ]:
def edit_distance(a: list, b: list) -> int:
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, start=1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

def cer(pred: str, gt: str) -> float:
    return edit_distance(list(pred), list(gt)) / max(1, len(gt))

def wer(pred: str, gt: str) -> float:
    return edit_distance(pred.split(), gt.split()) / max(1, len(gt.split()))

def load_ocr_baseline(path: Path) -> dict[int, str]:
    if not path.exists():
        return {}
    data = json.loads(path.read_text(encoding='utf-8'))
    items = data.get('results', data) if isinstance(data, dict) else data
    return {int(item['order']): item.get('text', '') or '' for item in items}

baseline_by_order = load_ocr_baseline(ocr_json_path)
rows = []
for rec in records:
    pred = baseline_by_order.get(rec['order'], '')
    gt = rec['corrected_text']
    rows.append({
        'order': rec['order'],
        'crop': rec['crop_path'].name,
        'pred': pred,
        'gt': gt,
        'cer': cer(pred, gt),
        'wer': wer(pred, gt),
    })

baseline_df = pd.DataFrame(rows)
display(baseline_df[['order', 'cer', 'wer', 'pred', 'gt']].head(20))
print('Mean CER:', baseline_df['cer'].mean())
print('Mean WER:', baseline_df['wer'].mean())
print('Exact match:', (baseline_df['pred'] == baseline_df['gt']).mean())

In [ ]:
baseline_metrics_path = DATASET_DIR / 'baseline_metrics.json'
baseline_metrics = {
    'manual_corrections': str(manual_md_path),
    'ocr_results': str(ocr_json_path),
    'mean_cer': float(baseline_df['cer'].mean()),
    'mean_wer': float(baseline_df['wer'].mean()),
    'exact_match': float((baseline_df['pred'] == baseline_df['gt']).mean()),
    'items': rows,
}
baseline_metrics_path.write_text(json.dumps(baseline_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', baseline_metrics_path)

## 5. Character dictionary

Словарь для обучения = `ppocrv5_cyrillic_dict.txt` + дополнительные символы из разметки (например `№`, `*`).

In [ ]:
chars = set()
for _, text, _ in label_entries:
    chars.update(text)

char_dict_path = DATASET_DIR / 'char_dict_avar.txt'
char_dict_path.write_text('\n'.join(sorted(chars)) + '\n', encoding='utf-8')
print('char dict:', char_dict_path)
print('chars:', len(chars))

## 6. PaddleOCR training repo

In [ ]:
if PADDLEOCR_DIR.exists() and not (PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yaml').exists():
    shutil.rmtree(PADDLEOCR_DIR)

if not PADDLEOCR_DIR.exists():
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR

expected_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yaml'
if not expected_cfg.exists():
    raise FileNotFoundError(
        f'PaddleOCR clone looks incomplete: {expected_cfg} not found. '
        'Delete /content/PaddleOCR and re-run this cell.'
    )

import os
os.chdir(str(PADDLEOCR_DIR))
!pip install -q -r requirements.txt

fix_train_deps()
verify_train_deps()

import paddle

paddle_version = tuple(int(x) for x in paddle.__version__.split('.')[:2])
if paddle_version[0] < 3:
    raise RuntimeError(
        f'Paddle {paddle.__version__} is too old for PP-OCRv5 training. '
        'Re-run section 1 (paddlepaddle-gpu==3.2.2), restart runtime, then re-run from section 2.'
    )

print('PaddleOCR OK:', PADDLEOCR_DIR)
print('paddle:', paddle.__version__, 'cuda:', paddle.is_compiled_with_cuda())

## 7. Pretrained recognition weights

Скачиваем train-веса **`cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams`** — ту же модель, что используется в `ocr_on_classical_crops.ipynb`.

In [ ]:
pretrained_dir = PROJECT_ROOT / 'pretrained' / REC_MODEL_NAME
pretrained_dir.mkdir(parents=True, exist_ok=True)
pretrained_path = pretrained_dir / 'cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams'

if not pretrained_path.exists():
    url = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams'
    print('Downloading pretrained...')
    urllib.request.urlretrieve(url, pretrained_path)

print('pretrained:', pretrained_path, pretrained_path.exists())

## 8. Training config

Шаблон: `configs/rec/PP-OCRv5/multi_language/cyrillic_PP-OCRv5_mobile_rec.yaml`.

In [ ]:
src_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yaml'
if not src_cfg.exists():
    raise FileNotFoundError(f'Config not found: {src_cfg}. Re-run section 6.')

base_dict_path = PADDLEOCR_DIR / 'ppocr' / 'utils' / 'dict' / 'ppocrv5_cyrillic_dict.txt'
base_chars = [line.strip() for line in base_dict_path.read_text(encoding='utf-8').splitlines() if line.strip()]
extra_chars = sorted(set(''.join(text for _, text, _ in label_entries)) - set(base_chars))
train_dict_path = DATASET_DIR / 'ppocrv5_cyrillic_plus_avar.txt'
train_dict_path.write_text('\n'.join(base_chars + extra_chars) + '\n', encoding='utf-8')
if extra_chars:
    print('Added to dict:', extra_chars)

config_text = src_cfg.read_text(encoding='utf-8')
config_text = config_text.replace(
    'character_dict_path: ./ppocr/utils/dict/ppocrv5_cyrillic_dict.txt',
    f'character_dict_path: {train_dict_path}',
)
config_text = config_text.replace('max_text_length: &max_text_length 25', 'max_text_length: &max_text_length 120')
config_text = config_text.replace('distributed: true', 'distributed: false')
config_text = config_text.replace('num_workers: 8', 'num_workers: 0')
config_text = config_text.replace('num_workers: 4', 'num_workers: 0')
# RecConAug на маленьком датасете часто ломается; для smoke-test отключаем.
config_text = config_text.replace('prob: 0.5', 'prob: 0.0')

config_path = PADDLEOCR_DIR / 'configs' / 'rec_avar_v5_smoke.yaml'
config_path.write_text(config_text, encoding='utf-8')
print('Using template:', src_cfg)
print('Config:', config_path)
print('char dict:', train_dict_path, 'size:', len(base_chars) + len(extra_chars))

## 9. Smoke-test training

`EPOCH_NUM = 1` только для проверки, что код обучения

Для реального обучения 30-50.

In [ ]:
os.chdir(str(PADDLEOCR_DIR))

fix_train_deps()
verify_train_deps()

EPOCH_NUM = 1
train_list = DATASET_DIR / 'train_list.txt'
val_list = DATASET_DIR / 'val_list.txt'
train_n = sum(1 for _ in train_list.open(encoding='utf-8'))
BATCH_SIZE = max(1, min(8, train_n))

cmd = (
    f'python tools/train.py -c configs/rec_avar_v5_smoke.yaml '
    f'-o Global.pretrained_model={pretrained_path} '
    f'Global.save_model_dir={OUTPUT_DIR} '
    f'Global.epoch_num={EPOCH_NUM} '
    f'Train.dataset.data_dir={DATASET_DIR} '
    f'Train.dataset.label_file_list=["{train_list}"] '
    f'Eval.dataset.data_dir={DATASET_DIR} '
    f'Eval.dataset.label_file_list=["{val_list}"] '
    f'Train.sampler.first_bs={BATCH_SIZE} '
    f'Train.loader.batch_size_per_card={BATCH_SIZE} '
    f'Eval.loader.batch_size_per_card={BATCH_SIZE} '
    f'Train.loader.drop_last=false '
    f'Eval.loader.drop_last=false'
)
print(cmd)
print('Лог train.py ниже (потоковый вывод). Первые строки могут появиться через 1–3 мин.')

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f'train.py failed with exit code {returncode}')

## 10. Export inference model + save to Drive

После обучения экспортируем модель в формат, который понимает `PaddleOCR(text_recognition_model_dir=...)`.

In [5]:
import os

In [6]:
os.chdir(str(PADDLEOCR_DIR))

best_ckpt = OUTPUT_DIR / 'best_accuracy'
if not best_ckpt.with_suffix('.pdparams').exists():
    found = sorted(OUTPUT_DIR.rglob('*.pdparams'))
    if not found:
        raise FileNotFoundError(f'No checkpoints in {OUTPUT_DIR}')
    best_ckpt = found[-1].with_suffix('')

if INFER_DIR.exists():
    shutil.rmtree(INFER_DIR)
INFER_DIR.mkdir(parents=True, exist_ok=True)

export_cmd = (
    f'python tools/export_model.py -c configs/rec_avar_v5_smoke.yaml '
    f'-o Global.pretrained_model={best_ckpt} '
    f'Global.save_inference_dir={INFER_DIR}'
)
print(export_cmd)

export_process = subprocess.Popen(
    export_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in export_process.stdout:
    print(line, end='')
if export_process.wait() != 0:
    raise RuntimeError('export_model.py failed')

train_dict_path = DATASET_DIR / 'ppocrv5_cyrillic_plus_avar.txt'


def sync_inference_character_dict(model_dir: Path, dict_path: Path) -> None:
    import yaml

    yml_path = model_dir / 'inference.yml'
    if not yml_path.exists():
        print('warn: no inference.yml in', model_dir)
        return
    if not dict_path.exists():
        print('warn: no train dict at', dict_path)
        return
    chars = [ln.strip() for ln in dict_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    cfg = yaml.safe_load(yml_path.read_text(encoding='utf-8')) or {}
    old_n = len(cfg.get('PostProcess', {}).get('character_dict', []) or [])
    cfg.setdefault('Global', {})['model_name'] = REC_MODEL_NAME
    cfg.setdefault('PostProcess', {})['name'] = 'CTCLabelDecode'
    cfg['PostProcess']['character_dict'] = chars
    yml_path.write_text(
        yaml.dump(cfg, allow_unicode=True, default_flow_style=False, sort_keys=False),
        encoding='utf-8',
    )
    print(f'inference.yml character_dict: {old_n} -> {len(chars)} chars')


sync_inference_character_dict(INFER_DIR, train_dict_path)

DRIVE_OUT = DRIVE_ROOT / 'models' / REC_MODEL_NAME
if DRIVE_OUT.exists():
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(INFER_DIR, DRIVE_OUT)

# checkpoint + config для §11 (predict_rec на export PP-OCRv5 ненадёжен)
ckpt_drive = DRIVE_OUT / 'checkpoint'
ckpt_drive.mkdir(parents=True, exist_ok=True)
for suffix in ['.pdparams', '.pdopt', '.states']:
    src = best_ckpt.with_suffix(suffix)
    if src.exists():
        shutil.copy2(src, ckpt_drive / f'best_accuracy{suffix}')

config_src = PADDLEOCR_DIR / 'configs' / 'rec_avar_v5_smoke.yaml'
if config_src.exists():
    shutil.copy2(config_src, DRIVE_OUT / 'rec_avar_v5_smoke.yaml')
if train_dict_path.exists():
    shutil.copy2(train_dict_path, DRIVE_OUT / 'ppocrv5_cyrillic_plus_avar.txt')

print('Inference model saved to Drive:', DRIVE_OUT)
print('Checkpoint saved to:', ckpt_drive)

FileNotFoundError: [Errno 2] No such file or directory: '/content/PaddleOCR'

## 11. Проверка кастомной модели (inference)

Прогон **только вашей** fine-tuned модели по crops: таблица `pred` vs `gt`.

**Важно:** экспортированный `inference.json` + `predict_rec.py` для PP-OCRv5 часто даёт мусор (`ӦÃӇ…`). Это известная несовместимость. Здесь inference идёт через **checkpoint** (`best_accuracy`) и **тот же Eval-пайплайн**, что при обучении (`infer_rec.py`).

Нужен `checkpoint/` на Drive — его пишет §10. Если его нет, один раз перезапустите **§9 → §10**.

**§1 не нужна.** После restart: **§2** → **§3** → **§11**.

In [ ]:
import os

FT_DIR = DRIVE_ROOT / 'models' / REC_MODEL_NAME
OUT_JSON = DATASET_DIR / 'finetuned_predictions.json'
LIST_FILE = DATASET_DIR / 'all_list.txt'
if not LIST_FILE.exists():
    LIST_FILE = DATASET_DIR / 'val_list.txt'

if not FT_DIR.exists():
    raise FileNotFoundError(f'Нет модели на Drive: {FT_DIR}')
if not LIST_FILE.exists():
    raise FileNotFoundError('Нет all_list.txt / val_list.txt — сначала §3.')

ensure_paddle_gpu()
fix_train_deps()
verify_train_deps()

if not PADDLEOCR_DIR.exists():
    print('Cloning PaddleOCR...')
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLEOCR_DIR)],
        check=True,
    )


def ensure_train_dict(list_file: Path) -> Path:
    dict_path = DATASET_DIR / 'ppocrv5_cyrillic_plus_avar.txt'
    drive_dict = FT_DIR / 'ppocrv5_cyrillic_plus_avar.txt'
    if drive_dict.exists():
        shutil.copy2(drive_dict, dict_path)
        return dict_path
    if dict_path.exists():
        return dict_path

    base_dict_path = PADDLEOCR_DIR / 'ppocr' / 'utils' / 'dict' / 'ppocrv5_cyrillic_dict.txt'
    if not base_dict_path.exists():
        base_dict_path = DATASET_DIR / '_ppocrv5_cyrillic_dict.txt'
        if not base_dict_path.exists():
            url = (
                'https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/main/'
                'ppocr/utils/dict/ppocrv5_cyrillic_dict.txt'
            )
            print('Downloading base cyrillic dict...')
            urllib.request.urlretrieve(url, base_dict_path)

    base_chars = [ln.strip() for ln in base_dict_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    extra = set()
    for line in list_file.read_text(encoding='utf-8').splitlines():
        if '\t' in line:
            extra.update(line.split('\t', 1)[1])
    extra_chars = sorted(extra - set(base_chars))
    dict_path.write_text('\n'.join(base_chars + extra_chars) + '\n', encoding='utf-8')
    print('Rebuilt train dict:', dict_path, 'size:', len(base_chars) + len(extra_chars))
    return dict_path


def ensure_rec_config(dict_path: Path) -> Path:
    local_cfg = PADDLEOCR_DIR / 'configs' / 'rec_avar_v5_smoke.yaml'
    drive_cfg = FT_DIR / 'rec_avar_v5_smoke.yaml'
    if drive_cfg.exists():
        text = drive_cfg.read_text(encoding='utf-8')
    else:
        src_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yaml'
        if not src_cfg.exists():
            raise FileNotFoundError('Нет config на Drive и нет шаблона PaddleOCR — запустите §8–§10.')
        text = src_cfg.read_text(encoding='utf-8')
        text = text.replace(
            'character_dict_path: ./ppocr/utils/dict/ppocrv5_cyrillic_dict.txt',
            f'character_dict_path: {dict_path}',
        )
        text = text.replace('max_text_length: &max_text_length 25', 'max_text_length: &max_text_length 120')
        text = text.replace('distributed: true', 'distributed: false')
        text = text.replace('num_workers: 8', 'num_workers: 0')
        text = text.replace('num_workers: 4', 'num_workers: 0')
        text = text.replace('prob: 0.5', 'prob: 0.0')
    text = text.replace(
        'character_dict_path: ./ppocr/utils/dict/ppocrv5_cyrillic_dict.txt',
        f'character_dict_path: {dict_path}',
    )
    local_cfg.parent.mkdir(parents=True, exist_ok=True)
    local_cfg.write_text(text, encoding='utf-8')
    return local_cfg


def resolve_checkpoint() -> Path:
    candidates = [
        FT_DIR / 'checkpoint' / 'best_accuracy',
        OUTPUT_DIR / 'best_accuracy',
    ]
    for ckpt in candidates:
        if ckpt.with_suffix('.pdparams').exists():
            return ckpt
    raise FileNotFoundError(
        'Нет checkpoint/best_accuracy.pdparams на Drive. '
        'Перезапустите §9 → §10 (обновлённый §10 кладёт checkpoint/ на Drive).'
    )


TRAIN_DICT_PATH = ensure_train_dict(LIST_FILE)
CONFIG_PATH = ensure_rec_config(TRAIN_DICT_PATH)
CHECKPOINT_PREFIX = resolve_checkpoint()
print('checkpoint:', CHECKPOINT_PREFIX)
print('config:', CONFIG_PATH)
print('train dict:', TRAIN_DICT_PATH)

eval_script = DATASET_DIR / '_run_checkpoint_rec.py'
eval_script.write_text(r'''
import json
import os
import sys
from pathlib import Path

import numpy as np
import paddle
import yaml

root = Path(os.environ['PADDLEOCR_DIR'])
os.chdir(root)
sys.path.insert(0, str(root))

from ppocr.data import create_operators, transform
from ppocr.modeling.architectures import build_model
from ppocr.postprocess import build_post_process
from ppocr.utils.save_load import load_model

dataset_dir = Path(os.environ['DATASET_DIR'])
list_file = Path(os.environ['LIST_FILE'])
out_json = Path(os.environ['OUT_JSON'])
config_path = Path(os.environ['CONFIG_PATH'])
checkpoint_prefix = os.environ['CHECKPOINT_PREFIX']


def normalize_label(text: str) -> str:
    return ' '.join(text.replace('\t', ' ').replace('\r', ' ').split()).strip()


config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
global_config = config['Global']
global_config['checkpoints'] = checkpoint_prefix
global_config['pretrained_model'] = None
global_config['infer_mode'] = True

post_process_class = build_post_process(config['PostProcess'], global_config)
char_num = len(post_process_class.character)

if config['Architecture']['Head']['name'] == 'MultiHead':
    out_channels_list = {
        'CTCLabelDecode': char_num,
        'NRTRLabelDecode': char_num + 3,
    }
    config['Architecture']['Head']['out_channels_list'] = out_channels_list
else:
    config['Architecture']['Head']['out_channels'] = char_num

model = build_model(config['Architecture'])
load_model(config, model)
model.eval()

transforms = []
for op in config['Eval']['dataset']['transforms']:
    op_name = list(op)[0]
    if 'Label' in op_name:
        continue
    if op_name == 'RecResizeImg':
        op = {op_name: dict(op[op_name])}
        op[op_name]['infer_mode'] = True
    elif op_name == 'KeepKeys':
        op = {op_name: {'keep_keys': ['image']}}
    transforms.append(op)

ops = create_operators(transforms, global_config)
use_gpu = paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0
if use_gpu:
    paddle.set_device('gpu')
print('use_gpu:', use_gpu)
print('checkpoint:', checkpoint_prefix)

items = []
for line in list_file.read_text(encoding='utf-8').splitlines():
    if not line.strip():
        continue
    rel, gt = line.split('\t', 1)
    gt = normalize_label(gt)
    img_path = dataset_dir / rel
    pred = ''
    if img_path.exists():
        with open(img_path, 'rb') as f:
            batch = transform({'image': f.read()}, ops)
        images = paddle.to_tensor(np.expand_dims(batch[0], axis=0))
        with paddle.no_grad():
            preds = model(images)
        post_result = post_process_class(preds)
        if post_result and post_result[0]:
            pred = normalize_label(str(post_result[0][0]))
    items.append({'file': rel, 'gt': gt, 'pred': pred, 'match': pred == gt})

payload = {
    'checkpoint': checkpoint_prefix,
    'config': str(config_path),
    'list_file': str(list_file),
    'items': items,
}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('OK', len(items), 'lines')
for row in items[:5]:
    print(row['file'])
    print('  pred:', row['pred'][:120] or '[empty]')
    print('  gt  :', row['gt'][:120])
''', encoding='utf-8')

env = os.environ.copy()
env.update({
    'PADDLEOCR_DIR': str(PADDLEOCR_DIR),
    'DATASET_DIR': str(DATASET_DIR),
    'LIST_FILE': str(LIST_FILE),
    'OUT_JSON': str(OUT_JSON),
    'CONFIG_PATH': str(CONFIG_PATH),
    'CHECKPOINT_PREFIX': str(CHECKPOINT_PREFIX),
})

print('Running checkpoint inference (Eval pipeline)...')
proc = subprocess.run(
    [sys.executable, str(eval_script)],
    env=env,
    cwd=str(PADDLEOCR_DIR),
    text=True,
    capture_output=True,
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr[-4000:] if len(proc.stderr) > 4000 else proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('Inference failed.\n' + (proc.stderr or proc.stdout))

result = json.loads(OUT_JSON.read_text(encoding='utf-8'))
df_pred = pd.DataFrame(result['items'])
print('Exact match:', df_pred['match'].mean())
display(df_pred[['file', 'pred', 'gt', 'match']])
print('Saved:', OUT_JSON)

## 12. Notes

- **Только посмотреть модель:** §2 → §3 → §11 (§1 не нужна).
- **Обучить заново:** §1 + restart → §2 … §10.
- OCR **новых страниц:** `ocr_on_classical_crops.ipynb` + `text_recognition_model_dir` = папка на Drive.